# 07-08_database_modeling_and_sql_queries

Create the city tables, load them into MySQL, inspect the schema, and clean up intermediate tables.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Set these in your environment before running:
# DB_USER, DB_PASSWORD, DB_HOST

In [ ]:
import pandas as pd

# Recreate or reuse df_global from your notebook
# cities table — just city_id and city name
cities_df = df_global[['city']].copy()
cities_df.insert(0, 'city_id', range(1, len(cities_df) + 1))

# facts table — city_id + the facts
facts_df = df_global[['city']].copy()
facts_df.insert(0, 'city_id', range(1, len(facts_df) + 1))
facts_df = facts_df.drop(columns=['city'])
facts_df[['country', 'latitude', 'longitude', 'population', 'scraped_at']] = df_global[['country', 'latitude', 'longitude', 'population', 'scraped_at']]

print(cities_df)
print(facts_df)

In [ ]:
# Connect to cities_db
from sqlalchemy.engine.create import create_engine


engine_cities = create_engine("mysql+pymysql://${DB_USER}:${DB_PASSWORD}@${DB_HOST}/cities_db")

# Push the tables there
cities_df.to_sql('cities', con=engine_cities, if_exists='replace', index=False)
facts_df.to_sql('city_facts', con=engine_cities, if_exists='replace', index=False)

print("Done!")

In [ ]:
from sqlalchemy import create_engine, text

# Connect without specifying a database
engine_no_db = create_engine("mysql+pymysql://${DB_USER}:${DB_PASSWORD}@${DB_HOST}/")

# Create a new database
with engine_no_db.connect() as conn:
    conn.execute(text("CREATE DATABASE IF NOT EXISTS cities_db;"))
    print("Database created!")

# Now connect to it
engine = create_engine("mysql+pymysql://${DB_USER}:${DB_PASSWORD}@${DB_HOST}/cities_db")

# Push your tables into it
cities_df.to_sql('cities', con=engine, if_exists='replace', index=False)
facts_df.to_sql('city_facts', con=engine, if_exists='replace', index=False)

print("Tables created in cities_db!")

In [ ]:
# cities table — already exists, leave it
# coordinates table
coords_df = facts_df[['city_id', 'country', 'latitude', 'longitude']].copy()

# population table
population_df = facts_df[['city_id', 'population', 'scraped_at']].copy()

# Push both to cities_db
coords_df.to_sql('city_coordinates', con=engine_cities, if_exists='replace', index=False)
population_df.to_sql('city_population', con=engine_cities, if_exists='replace', index=False)

print("Done!")

In [ ]:
pd.read_sql("SHOW TABLES;", con=engine_cities)

In [ ]:
with engine_cities.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS city_facts;"))
    conn.commit()
    print("Done!")